# **Pythonプログラミング演習 2026年度 実践課題**

## **課題概要**

本課題では，**「オセロをプレイするAI」** を自作します．

独自の `Player` クラスを実装し，[Webシステム](https://othellopy.com) より提出してください．

## **課題提出**

**【中間提出】**

**提出物**：
* 実装したオセロAI．[Webシステム](https://othellopy.com) より提出

**締切**：**08/03**（第13・14回講義日） **23:59**\
**採点**：提出点のみ

**【最終提出】**

**提出物**：
* 実装したオセロAI．[Webシステム](https://othellopy.com) より提出
* 実践課題レポート（後日指示）

**締切**：**09/30** **23:59**\
（早期提出：08/31）

**採点**：
* 提出点
* 性能点：提出したオセロAIの性能に応じて採点
    * 「秀」の目安：中級レベルのCPU `IntermediatePlayer` に確実に勝てる性能であること
    * 最終提出の性能に応じて加点あり
* 実践課題レポートで工夫点を詳しく説明している場合加点あり

## **問い合わせ**

* よくある質問は [FAQページ](https://othellopy.com/faq) にまとめています．

不明点は，以下の方法でお問い合わせください．
* 授業中，教員もしくは TA に質問する（現地・オンライン）
* [TAのメーリングリスト](https://mail.google.com/mail/?view=cm&fs=1&to=python_pp_ta@ml.naist.ac.jp) `python_pp_ta@ml.naist.ac.jp` に質問内容をメールする

---
## **ソースコード本体**

### **必要なパッケージのインストール**

本課題では Python でオセロを実装するためのライブラリ `othellopy` を利用します．

In [9]:
!pip install -U othellopy

### **必要なライブラリのインポート**

In [10]:
from othellopy.core import Board, Cell, Move
# Board: オセロ盤面を表す型．8 x 8 の2次元配列．
# Cell: 各マスの状態を表す値．EMPTY（0), BLACK (1), WHITE (2)．
# Move: 石を置く位置を表す型．(row, col) のタプル．

from othellopy.players import BasePlayer
# BasePlayer: 自作プレイヤーの基底クラス．get_moves などの補助メソッドを提供.

### **<font color="red">【提出対象】自作AI `MyPlayer` の作成</font>**

以下の class `MyPlayer` が制作・提出対象です．\
自分なりにアレンジして強い「オセロAI」を作成してください．

#### **サンプル**

```
class MyPlayer(BasePlayer):
    def next_move(self, board: Board) -> Move:
        moves = self.get_moves(board)
        return moves[0]
```

#### **実装時の制約**

* 利用できるライブラリは標準ライブラリのみ（`random`, `math` 等）
* 外部ライブラリ（`numpy`，`scipy` 等）利用NG．
* 外部データ参照NG（`requests`，`os` 等）
* `next_move()` の実行時間制限は2秒（重すぎる処理はNG）
* 生成AI利用可（出力を鵜呑みにせず，有効に活用してください）

### **解説**

* `next_move()`
  * 現在の盤面を受け取り，次に打つべき１手を決定する．
  * 返り値：次に打つ手の座標（例：`(0, 0)`）
* `board: Board`
  * 盤面を管理する二次元配列
  * 左上が `board[0][0]`
  * 例：`board[3][5]` = 上から4段目，左から6列目
  * `0` = 空白，`1` = 黒，`2` = 白
* `self.get_moves(board)`
  * 現在の盤面での有効手を一覧で取得する．
  * 例：`[(2, 3), (3, 2), (4, 5), (5, 4)]`

### **提出方法**

[Webシステム](https://othellopy.com) より以下のコードセルで作成したオセロAIをコピー＆ペーストして提出

---

**<font color="red">↓ 以下のコードセルを実装・提出</font>**

In [ ]:
from othellopy.core import Board, Cell, Move
from othellopy.players import BasePlayer


# 提出時は，本コードセルをコピーして提出する
class MyPlayer(BasePlayer):
    BOARD_INDEXES = range(8)
    SEARCH_DEPTH = 5
    PROBCUT_MIN_DEPTH = 3
    PROBCUT_MARGIN = 0.16
    PROBCUT_SHALLOW_DEPTHS = (0, 0, 0, 1, 2, 1, 2, 3, 4, 3, 4, 3, 4, 5, 6)

    DIRECTIONS = (
        (-1, -1), (-1, 0), (-1, 1),
        (0, -1),           (0, 1),
        (1, -1),  (1, 0),  (1, 1),
    )
    PATTERN_INDEXES = {
        "diagonal8": (
            (0, 9, 18, 27, 36, 45, 54, 63),
            (7, 14, 21, 28, 35, 42, 49, 56),
            (63, 54, 45, 36, 27, 18, 9, 0),
            (56, 49, 42, 35, 28, 21, 14, 7),
        ),
        "edge2X": (
            (9, 0, 1, 2, 3, 4, 5, 6, 7, 14),
            (9, 0, 8, 16, 24, 32, 40, 48, 56, 49),
            (49, 56, 57, 58, 59, 60, 61, 62, 63, 54),
            (54, 63, 55, 47, 39, 31, 23, 15, 7, 14),
            (14, 7, 6, 5, 4, 3, 2, 1, 0, 9),
            (49, 56, 48, 40, 32, 24, 16, 8, 0, 9),
            (54, 63, 62, 61, 60, 59, 58, 57, 56, 49),
            (14, 7, 15, 23, 31, 39, 47, 55, 63, 54),
        ),
        "triangle": (
            (0, 1, 2, 3, 8, 9, 10, 16, 17, 24),
            (0, 8, 16, 24, 1, 9, 17, 2, 10, 3),
            (7, 6, 5, 4, 15, 14, 13, 23, 22, 31),
            (7, 15, 23, 31, 6, 14, 22, 5, 13, 4),
            (63, 62, 61, 60, 55, 54, 53, 47, 46, 39),
            (63, 55, 47, 39, 62, 54, 46, 61, 53, 60),
            (56, 57, 58, 59, 48, 49, 50, 40, 41, 32),
            (56, 48, 40, 32, 57, 49, 41, 58, 50, 59),
        ),
    }
    PATTERN_SIZES = {"diagonal8": 8, "edge2X": 10, "triangle": 10}
    WEIGHTS = (
        0.877998471, -0.609513104, -1.20360065, 0.317599475, -0.147198424, 0.243788362, 0.040882796, -0.400356025,
        -0.487287223, -0.119104415, 0.359493762, -0.0946342424, -0.277470052, 0.954106092, -1.27141988, -0.556041002,
        1.20066679, -0.730736673, 0.960250616, -0.318587005, -1.05440593, -1.17217064, 1.44991183, -1.08265722,
        -1.22989428, 1.59414148, -0.51240164, 0.253132552, 0.234904468, -0.169593289, 0.69953841, -0.190667734,
        0.194870219, 0.179621369, -0.311098009, -0.0557038523, 0.291046143, -0.834287107, 1.95058739, -1.49098277,
        0.45816946, 0.161768749, -0.31763494, 0.276269734, 0.432592154, 0.446508706, -1.1165328, -1.75409448,
        0.550034463, 0.178742155, 0.18127282, 0.38963753, 0.28794983, -0.816410303, 0.324321896, -0.995132029,
        0.440137506, 0.352155656, 0.825089633, -0.642814219, -0.346161991, -1.14827001, 0.689906359, -0.181644499,
        0.2736485, 0.542473614, -1.55398393, 0.0594785251, -0.656590044, -0.146631435, -0.17267476, 0.854212642,
        -0.985429049, -0.500189483, -1.45098567, 0.176609561, 0.526618481, 0.483834773, 0.705321074, 0.526528955,
        -0.919260621, 0.446625859, -0.543612659, 0.384040713, -0.301126719, -0.920317292, 1.59222448, -1.16093826,
        0.518311739, 0.239203215, -1.49929655, -0.830533803, -0.288776129, 1.00002372, 0.345835447, -0.097938396,
        -2.4578023, -2.24901128, 0.245968953, 0.0401698537, 0.283357948, -0.952127039, 1.2469523, -1.77569127,
        0.634282947, -2.87696648, -0.981946766, 0.547502279, 0.431264102, -1.76494122, 0.173367813, 1.54281199,
        -1.23792791, -1.55196118, -0.915029824, -0.286652595, 0.2056306, -0.0534627885, -0.732607245, 0.75359571,
        -0.554344893, 0.459221452, 0.116468377, -0.477375805, 0.0662493631, 0.279409766, 0.611403763, 0.718376994,
        -0.512283504, 0.609909773, -0.529900789, 0.0525996312, 0.328570455, -0.354128748, 0.196535826, -0.219426647,
        -2.11007571, -1.13907826, 0.0705398992, 0.468658775, -0.128773078, -0.497493744, 0.413283974, 0.267521501,
        0.820222378, -0.0115940394, 0.0286926515, -0.0501629002, -0.708179355, -0.759693265, 0.352468729, 0.658496797,
        -1.05147052, -2.65756726, -0.867285967, -0.869638324, -0.692349672, 0.488590121, -1.12383473, 0.608238041,
        -3.17771697, 0.0383900478, 0.5264799, 0.66974026, -0.398304194, -0.868673742, 1.29336119, -2.27166677,
        0.106895737, -0.787223458, 0.586030304, -0.278997421, 0.106911324, -0.562530637, -0.894575596, 0.347732127,
        -2.59544492, 0.519491196, -0.133727074, -0.8076002, -0.344378918, -0.468670011, 0.0416513421, -1.14138126,
        0.0342367031, -1.76723433, -0.292882413, 0.531401694, 0.323342711, -1.11589313, 0.396852702, 0.617043257,
        -0.227520317, -0.641157508, -0.0530696847, 0.0131349713, 0.405873239, 0.0839118659, -0.937733889, -0.79688549,
        -0.430902779, -0.661940992, -0.706246674, -0.0548673309, -2.5229342, -0.58525914, 0.759495437, -0.905770481,
        -0.678119421, -0.244688153, -0.476578534, 0.389288992, 0.134338409, -0.127702385, -0.775632679, 0.853273451,
        -1.20517027, -0.845670581, -2.01915741, -1.17777717, -0.565559566, 0.659383833, 0.970590949, 0.570956409,
        -0.554051816, 0.376849025, -0.1261684, -0.0456960425, -0.0160913244, -0.957402706, -0.671032488, 0.552811742,
        0.977993131, 0.115494125, 0.4515737, -0.571180701, -0.34766832, 0.216985062, -0.833115995, 0.49776724,
        -2.38665771, 0.910461605, -1.55967259, -0.549075484, -0.541783035, 0.026070198, 0.957200348, -0.11200954,
        -1.28923452, 0.326571226, 0.663004816, 0.00285580475, 0.0424677432, -0.789683759, -2.06211901, -0.589474976,
        -0.00405926816, -0.0555754341, -0.0397314578, 0.017306041, -0.0545230694, -0.266393602, 0.0729418099, -0.260049909,
        -0.140641868, -0.393380165, -0.026235003, -0.206615031, 0.0394535363, -0.0661846399, -0.14142397, -0.259716988,
        -0.735440135, -1.12836111, 0.330389738, -0.538958728, -0.353909999, -0.736204267, -0.494275451, -0.717202783,
        0.178154156, -0.45856604, -0.381053865, 0.160035625, -0.811162472, 0.074734427, -1.91599727, -0.646245122,
        -1.61287212, 2.1885078, -0.590166986, -0.975490332, -2.73520875, 0.588676751, 1.00235736, -1.9335444,
        -1.15235257, -0.743612885, -0.491796315, 0.116209, -0.742592573, -2.05503654, -0.661720932, 0.134005621,
        -0.559501112, -1.29590762, 0.212702826, 0.478181034, -0.366392553, -0.674042881, -0.565975726, -0.00331672188,
        0.534162521, 0.823142886, -1.62813044, -0.199954122, 0.395785272, 0.47202158, 0.615701318, 0.74912715,
        1.17306387, -0.802574635, 0.0924043581, -1.97920716, 0.43237114, 0.363466471, -0.51746273, -0.0864515081,
        0.274981827, 0.392493278, -1.7076236, 0.236740708, 0.0822909027, -0.568947375, -0.0305661261, -0.47271961,
        -0.595588744, -1.25364935, 0.607452095, -1.24809086, 0.177949145, 0.378572971, -0.830751836, 0.180015177,
        0.678289592, 0.0577746406, -0.791571081, 0.214016229, 0.26627022, 1.05847538, -1.07533181, 0.2377,
        -1.41625214, 1.6225338, -0.747112751, 0.201451436, -3.82081151, 0.248301238, 0.827008486, 0.442694783,
        -1.36086607, -0.88430804, 0.401467264, -0.0260485429, -1.565557, -1.85579896, -0.895299315, 0.0825887769,
        -1.21312702, 1.30133998, -0.361063898, 0.486500353, -1.92865384, -0.201541856, 0.733240545, -1.17970371,
        0.371534795, -0.654162169, -0.208947629, -0.182347313, -2.04037237, -0.00711173285, -1.1568948, -1.19309342,
        1.19299185, -2.25156403, -0.427094758, -0.394871742, 0.173588976, 0.719811738, 0.179660037, 0.252821058,
        -0.99039948, 0.218442842, -1.6233474, -0.757809758, -0.996255219, 0.329673916, 0.336543351, 0.095129028,
        0.618704319, 2.82081151, -0.210613206, 0.266365081, 0.178869411, 1.1582427, 0.245286047, -0.657775819,
        0.17678979, -0.652246058, -0.00591628067, -1.10223782, -0.824424267, 0.329032123, -0.234774515, 0.032715898,
        -0.750013351, -0.94385016, -0.57083261, 0.316776782, -0.211506188, -0.986638665, 1.62675786, 0.632629156,
        -0.121926025, 0.447422057, -0.0907138735, -0.323053747, -0.0920597315, 0.0995978266, 0.48939243, 0.181688458,
        -0.102751374, 2.51035404, 0.691405654, -1.37754118, -1.14199913, 1.80635953, 1.83654714, 0.252486199,
        0.0512380153, -1.25695753, -1.07952261, -0.615427077, -1.26574898, 1.02247751, 0.347057372, -0.522616684,
        -2.5382967, -1.36041284, 0.378224701, 0.0957987607, -0.574755788, -0.733121693, -0.378464848, -1.04142702,
        0.405196965, 0.924850941, 0.010637138, -0.630560696, -1.29504478, 0.57551074, -0.880880594, -2.55758977,
        0.642318606, 0.839872122, 1.53455257, -1.98245609, -0.917755127, 2.30995393, 2.86356401, 0.0216337852,
        0.267339438, -0.826411784, -1.64400697, 0.609214067, -0.129512519, 1.57182741, 0.376299113, -2.44175029,
        0.336029291, -1.6117065, 0.504036248, -0.414124221, -0.778188109, -1.01236308, 0.172403783, 0.214079261,
        0.63511616, 0.751528859, -1.08624804, 0.168404341, 0.246865228, -0.745024621, 0.329970658, -1.41788578,
        -1.094679, -1.03502655, -0.265510291, 0.182125404, -0.974101245, -0.82774359, -0.661031127, -0.476390719,
        -0.186765403, 0.787509978, 0.29996866, -0.125289738, 0.16742073, 0.103157543, 0.790137827, -0.15477109,
        0.501927435, 0.950249016, 0.825623035, -0.78472507, -0.945768774, 1.6892854, 1.85207558, -0.0154449539,
        0.460970491, 0.288891852, -0.677329361, 0.217024192, -0.0185300577, 1.32104075, 0.238359898, -1.68789577,
        -0.123753607, -0.784712493, -0.630830348, -0.62481153, -0.400841683, -0.375079662, -0.345012754, -0.131670192,
        0.242360875, -0.661498666, 0.0206138063, -0.163955808, -0.727982163, 0.0950714871, -0.602110267, -0.521193326,
        0.0786058307, 2.36322451, 0.17465353, 0.324525952, 0.278060228, 0.460261136, 0.653231561, 0.4745543,
        -0.685526848, -0.796442211, -1.64938939, 0.222822055, -2.92650342, 0.164838821, 0.104061872, -0.849945068,
        0.0628725588, 0.158170089, -0.324643701, 0.856935918, 0.117251761, -0.18252103, 0.501282275, 0.202289626,
        0.224779293, 0.481253147, -0.293984562, -1.50022852, 1.35611653, 0.0227289647, 0.308412224, 0.573277891,
        0.43241173, 0.426835209, 0.577161014, -0.683308721, -0.263216794, 0.550179303, -1.17724097, 0.771231055,
        -0.747695267, 0.409484535, -1.40643311, 0.828505158, -1.1160661, 0.710150659, 0.657119274, -0.1369721,
        0.540236473, -0.334358126, 0.159802511, -0.512487411, -0.296235502, -0.488440067, -0.44559738, -0.244630322,
        0.476119995, -0.470335484, 0.6683231, 0.579788387, -0.01619583, -0.235258505, -0.224284261, 0.48393932,
        -0.787730455, 0.330245018, 0.577728331, 0.394332558, 1.44604611, 1.04173684, 0.404023588, -0.0429471433,
        -1.03683841, -0.545123339, -0.927635908, 0.779579222, 0.265994012, -0.209460065, -0.642947555, 0.958295107,
        -0.35138911, -0.0836724639, 0.715235531, -0.938808203, -0.0708173439, 1.44054806, 0.00691584637, -0.574083209,
        -1.43776071, 0.290919125, -0.521091044, -0.647858679, 0.529488802, -0.13137491, 0.759037852, 0.932716846,
        0.0948010609, -0.24133645, -1.07004201, -0.363172889, -0.104643412, -0.480707735, -0.41658932, -0.207913339,
        0.67349267, 0.159245595, -0.0829768479, -1.4963783, -1.66911292, -0.0973879993, 1.41351199, -1.3238498,
        -0.319313973, 0.519782722, 0.589146376, -0.5247491, 0.588879228, -0.101541616, -1.08306861, 0.495161444,
        0.136485502, -0.672538877, 0.86827749, -1.08654904, 0.0930980146, -0.836412966, -0.039787285, -0.126297042,
        0.752172351, -2.10714126, -0.233882487, 0.62408632, 0.495573789, -0.324163049, -0.651927769, -0.256210506,
        -0.827474058, 0.204363972, -0.0490640923, 0.203714207, 0.284794122, 0.568223178, -0.82109046, -0.381381929,
        -0.686022162, 0.0645315647, -0.737518072, 0.142953232, -0.665766358, -1.23299217, 0.66949904, -1.64115655,
        0.0235946309, 0.455604613, -0.30664444, -0.0276034195, 0.221570373, 0.518434525, -0.914548397, -0.277000844,
        -0.835541725, -0.352115422, -0.484660774, 0.166188523, 0.684166968, -1.26403928, 0.0735616237, -0.108852819,
        1.33410358, -0.53112942, 0.0637537614, 1.26216912, 0.591968894, -0.759469807, -0.289693773, -0.450548738,
        -0.422677219, 0.193359688, 0.089485608, 0.507281303, -0.76396805, -0.461754203, -1.04720461, -0.57935977,
        -0.507272422, -0.431631327, -0.151342988, -0.575663805, -0.526488602, 0.84853524, -0.975426912, 0.687062442,
        -0.241054073, 0.232049212, -0.773638666, -0.269738972, 0.435812533, 0.942839324, 0.58207649, 0.717868209,
        -0.111458197, -0.735606074, -0.376655906, 0.395798355, -0.319950312, -0.0628379732, -0.0600820221, 0.407547086,
        -0.143998623, 0.384178817, 0.264193565, -0.961478531, 0.452708513, 0.929772854, -0.51461488, -0.142642587,
        0.0676316693, -1.44883943, 0.28330937, -1.49532914, 0.498583943, 0.600070775, -0.561381042, 0.736124933,
        1.07129836, -1.81350911, 0.0285273511, -0.148773625, -1.10521889, 0.0428969078, 0.321760684, -0.0449307039,
        -0.305237055, 0.66853106, -0.871094465, -0.877244055, -1.83971655, 0.352438033, 0.492600828, -1.64447987,
        -0.972742796, 0.456406564, 0.868400037, -0.654892921, 0.0334808454, -1.2404176, 0.0683549717, -0.373669863,
        -0.150982589, -0.677238107, 0.417371899, 0.396915317, 0.309817672, -0.14060472, -0.0818739906, 0.5119102,
        0.754488468, 0.313416481, 0.733213127, 0.0146992616, -0.526150942, -0.103415683, 0.634077549, -0.658230841,
        0.764332056, -0.11111328, 0.766855121, -1.01367021, 1.23571789, -0.066986233, 0.242811605, -0.436564744,
        -2.52078962, -0.298397213, 0.879729688, 0.276794463, 0.583471358, -0.624270856, -1.14843225, -0.362430543,
        0.210062087, 0.114473291, 0.540710032, 0.124617852, 0.0536038093, 1.04205525, 0.906277835, -0.969167292,
        1.38235831, -0.500650287, -0.738886297, -0.766533554, 0.616212726, -0.604392529, 0.0321660414, -0.0701673105,
        -0.563182712, 0.133970395, -0.656432629, -1.66711581, 0.874257803, -0.377333999, 0.777202725, 0.646351695,
        0.379166871, 0.926245332, -0.674800038, 0.143006369, -0.765630782, -0.71807003, -0.746910036, 0.130159631,
        0.485927522, -0.803833902, -1.08542621, 0.711306691, -0.141590267, 0.395302892, -0.754945219, 0.832798958,
        -1.17961669, -0.584785819, 0.361580074, 0.525690615, 0.5937047, -1.17605793, 0.112824813, -0.867293,
        1.8956157, 0.151754916, 0.43107304, -0.668457985, 0.902244985, -0.0903917626, -0.015800748, -0.0568031259,
        -0.123060942, -0.280415177, 0.333057016, -0.0414487049, -0.0124844518, 0.297530532, 0.672831714, 0.137907192,
        -0.126127258, 0.479263186, -0.574098527, -0.595574796, -0.320063561, -0.749983966, -0.0477842428, 0.195351973,
        -0.28514266, -1.90201592, 0.625623047, -0.313939452, -0.0142854117, 0.252995908, -1.77289844, -0.361539364,
        -0.767359614, -0.0862615779, 0.114922836, 0.238502294, 0.20134899, -1.51367259, 0.43745169, -1.23921239,
        0.839409709, -1.12206697, 0.204419866, -1.0097115, 0.0904765651, -0.0710519776, 0.701847196, 0.662587643,
        -0.413053423, 0.638311386, 0.47140485, 0.333110958, -0.699323118, -0.0793906823, -0.726863325, 0.634370387,
        -0.456983149, -0.850920141, 0.178680882, -0.694587469, 1.04776168, -1.30276811, 1.44052815, 0.0497283228,
        -0.501910269, 0.524305165, -2.1073873, 1.48281145, -1.65860522, -1.4040463, 0.922569513, -1.2654593,
        1.37892532, -0.220196694, 0.19727172, 0.287758201, -0.67993623, -0.142279729, 1.3779074, 0.552129328,
        -1.10823989, -0.521427989, 0.340839624, 1.07888782, -0.266272932, 0.429766625, -0.634559393, -0.348074794,
        -2.0816288, 0.161454275, -0.303837717, -1.32653642, 0.840151668, -0.234079003, -0.478108108, 0.184990421,
        0.577293396, 0.637737572, -0.957810104, -0.952339232, -4.51368475, -0.585936785, 0.0117638782, 0.0415740423,
        -0.898413599, 0.340161145, 1.13199639, -0.0239012986, 0.226397216, -1.50631046, 1.52073491, 0.351787001,
        -1.78429127, -1.62668312, -0.227130786, 3.30748081, 0.122920193, -1.15772653, 0.538879871, -2.22668028,
        -0.034241721, 0.336754888, 0.032974761, 0.367264003, -0.516394556, -0.364009082, -0.198603228, 0.943265736,
        -0.273817748, -0.245182663, 0.33791405, 0.712626159, 0.0375898182, -0.515173614, -0.648464024, -0.0973940343,
        -1.33043909, 0.315842599, -1.91085136, -1.51010108, -0.804578185, -1.02750921, -1.04327214, 0.550779581,
        0.0244794022, -1.21870053, -0.339322656, 2.45963073, 0.296956599, -0.0420143194, -3.13651991, 0.0540511124,
        0.045419734, 0.143738806, -3.35476971, 0.340340823, -0.00575005077, -1.43271673, -1.36707282, -1.89856923,
        -1.22467554, -0.419098079, 0.242463559, -0.787717462, 0.424423665, 0.0257387441, -1.34252679, 0.44014582,
        -1.43705904, -0.177116558, 1.00321066, 0.25304231, -1.03084064, -0.105613753, -0.260593265, 0.115847193,
        -1.57535481, -1.79261982, -0.387561798, 1.66600323, 0.389534831, -0.678237557, 0.27729702, -0.435039699,
        0.0210073814, -0.0982735902, -0.649423063, 0.573600888, -1.57978427, -1.01643384, 0.150007606, 0.785542488,
        -0.218511879, 0.412190527, -0.595854223, -1.33234859, 0.135730565, -0.148146331, -2.5654695, 0.175715074,
        -0.281009316, -0.326054722, 0.256859303, -0.698096275, 0.443837136, -0.848428547, -1.22350574, 0.361427605,
        -0.193670139, 0.240318, -0.549327672, -1.82823193, 0.386667252, 0.0390094966, -0.499426335, 0.923731685,
        -1.50287366, -0.25747025, -2.55630541, -0.168204054, 0.779177487, -0.644539595, -0.36200425, -2.90165973,
        -0.419937521, -0.373992354, -0.482086569, -2.1035378, 0.458711326, -1.07625413, -0.276638508, -0.00434818491,
        0.105142869, 0.341319293, -1.10054338, 0.472190797, 0.417684495, 0.138134718, 0.896649837, -0.405852556,
        -0.383582652, -1.04208827, 0.735591471, -1.83995116, 0.0592219681, -0.607407331, 0.582039893, 0.336257726,
        0.734871745, -0.0734439418, 0.502445579, 0.132063389, -0.553909242, 0.79627651, -2.2253077, 0.17912969,
        -0.0748717785, 0.247124925, 0.37641713, -1.11678815, -0.219679743, 0.346596837, 1.03012753, -1.12401402,
        0.678041399, 0.0678958744, 0.526245356, -2.24343872, 0.352203637, 0.209893227, 0.994425893, -0.33123365,
        0.33818391, 0.210091189, 0.0947950855, 0.0141282231, -0.651730001, 0.265519649, 1.40476179, -1.33822751,
        -0.0646205395, 0.167254001, -0.11369133, 0.0893909484, -0.797586381, -0.398622125, -1.00793695, 0.273728132,
        -0.534312963, -0.648770511, 0.313577771, -0.100542255, -0.16389814, -0.180518836, -0.517826498, -0.482327342,
        0.0513155684, 0.874316216, -0.301580191, -3.50213361, -0.674995124, 0.490486473, -4.57468176, 0.198140204,
        -2.04349303, -0.593873262, -0.444982857, 0.208050132, -0.242047116, -0.757241309, -0.284568638, -0.181817204,
        -0.290780544, 0.0112708537, 0.208396628, -0.291825801, -0.889996648, -0.137975082, -0.00531511242, 0.471068829,
        -1.18091583, -1.47914004, -1.11101341, 0.169741347, 0.3113662, 1.19790566, 0.223977298, 0.143970475,
        0.341011703, 0.247332886, 0.475057423, 0.549969494, -0.289809883, 0.0339493565, -0.306599021, 1.38399804,
        -0.774258673, -0.119242214, -1.71776772, 0.980673492, 0.379181296, 0.0171114914, 0.618105531, -0.823106587,
        0.619264781, -1.03177214, -0.201476857, -0.889626503, -0.995973408, 0.89388895, 0.477469116, 0.114686541,
        -0.60818851, -0.641377687, -1.65561211, -0.0731181279, 0.135278404, 0.777371109, -0.11793337, 0.0574663766,
        -0.702261686, -0.0233548153, -3.17574859, -0.716396272, 0.802821279, 0.104635261, -0.801789761, 0.672885776,
        -0.890034378, 0.0659393221, -0.199854344, -1.19528461, 0.971738279, -1.75464451, 1.28356194, -1.4319396,
        0.309872538, -0.171251953, 0.56812191, 1.0416317, 0.140681699, -0.107579827, 0.487832904, -0.750187695,
        1.47501361, -0.984370589, 0.667868137, -0.149947405, 0.841515481, -0.577322006, 0.143823907, -0.0954191089,
        -0.700307012, -0.366010249, -0.331087589, -0.333413422, 0.402841359, -0.753868699, 0.315768003, 0.165018663,
        -0.193998098, -0.458516032, 1.24371934, -0.486555338, -0.800151825, -0.376208961, 0.963763654, -0.234782189,
        0.219078049, -0.953057766, 0.105772406, 0.293804049, -0.356104046, 0.606076241, -0.408246279, 0.163285181,
        0.520983517, 0.0992105007, -0.788475037, 1.18403077, 0.380660564, -0.673374593, 0.657223225, 0.0156728905,
        0.419750154, 0.376155883, -0.74380511, 0.521529377, -0.103686869, 0.50972873, 0.528179288, -1.10826159,
        -0.488733649, -0.0225011837, 0.94514817, -1.23883057, -1.25421059, -0.418028414, 1.5965625, 0.516896904,
        -0.1579597, 0.428134233, 0.210919514, -0.131714329, -0.00328860688, 0.281293154, 0.378191292, -0.463526756,
        -0.00568181975, 0.799710095, -1.27891624, -0.261348307, 0.15408884, 0.807906747, -1.0070926, -2.07077575,
        0.616357267, 0.247951299, -0.374976814, -1.73962224, -0.44358182, -1.6146071, 0.867616594, 0.853107095,
        -1.75117707, -0.900186777, -0.0489889421, -0.967412293, -1.526618, 0.722085476, 0.589548647, -2.40762353,
        -1.41363847, -0.120892607, -1.07536387, -0.266551465, -1.10665858, 0.592757285, -0.00586741883, -0.305323303,
        1.16701245, -0.00327074947, 0.466016144, 0.0330359675, 1.02253604, -0.294655442, -0.694832265, 0.192745402,
        -0.383791745, -1.18522, 0.502069175, 1.02776003, 0.110245839, 0.469060004, 0.092981793, -2.46353436,
        -0.49753055, -0.0182089843, -1.75716865, 0.681881905, 0.860976875, 0.0602475889, -0.176048011, -0.794126928,
        0.0471732989, 0.359698266, -0.847245514, 0.0421158783, 1.01632476, 0.708498895, -0.924024403, 0.468650937,
        -0.0539669208, 0.175206006, -0.0942379907, 0.949262917, -0.268756837, -1.53798699, 0.268284321, 0.551895618,
        -1.27064443, -0.653488159, -0.600630701, 0.249043047, 1.41088986, 0.462879479, 0.471838892, -0.82532382,
        -2.56882715, -0.259806365, 0.483624935, 1.05005813, -0.47925967, -0.697181702, -0.575040877, -0.687523484,
        0.381134123, 0.0531651899, 0.595324039, -0.599686027, -0.160987556, -0.866054833, -2.30333567, 0.311684906,
        -0.226978049, 0.579499304, -0.20014523, -0.280032992, -1.15397191, -0.834938645, 0.547280788, 0.88654232,
        0.738881826, 0.492922515, 0.537168145, 0.0550155193, 0.339750022, -1.29219759, -0.439296812, 0.0484170392,
        -0.866002262, 0.480948567, 0.754416466, -0.412234515, 0.738502622, -2.38009834, 0.322274953, -0.678346634,
        1.45296788, -0.297474384, 0.362548798, -0.602870941, 0.328235209, 0.358630598, 0.356179237, -0.367888689,
        -1.40187287, 0.119853899, -0.544537067, 0.103005193, 0.848547518, -0.614984393, 0.758868873, -0.788352251,
        -2.38779759, 1.67944705, -1.75947225, 0.141095176, 0.318257689, 0.587004662, 0.385592222, 0.414590091,
        0.38335523, -0.714233816, -0.859794199, 0.662712574, -0.728350282, -0.386250496, 0.426276594, -0.871952236,
        -1.15372133, 0.859289944, -0.0844485238, 0.845391154, -1.50735867, 0.845549226, -0.552666545, -0.172845572,
        0.584573686, -0.810892403, 0.176621854, 0.140742272, -0.0260448027, 0.044927597, 0.482919395, 1.09056997,
        -1.30364716, 0.268366516, 0.428981543, 0.202664822, -0.324300051, 0.445618808, -0.237301379, -0.589050353,
        0.754157603, 0.608172178, 0.969595313, 0.478137374, -0.532929301, 1.01225758, -0.303252995, 0.437022507,
        -0.269364685, -1.12377584, 0.383488536, 0.0287557114, -0.0175185148, -0.385180175, 0.252285779, 0.119236894,
        -0.248098195, -0.244630918, 0.0630791783, -0.0583548881, -0.376655489, -0.130606949, 0.145380512, -0.221326366,
        -0.546745658, -0.180351377, 0.464986771, 1.04059196, 0.281449288, -1.14612889, 0.204011187, 0.65213567,
        -0.839920998, -3.5950284, -0.350196719, 0.451602936, -2.99794769, 0.0612084419, 0.178151831, -0.767768025,
        -1.62441409, -0.236721396, -0.172722101, -0.394941926, 1.02468443, -1.68176246, 0.745425045, 0.22701709,
        -1.94898403, -5.13537502, 0.107032351, -0.204489887, -2.94439101, -0.226112202, 0.633511007, -2.20201898,
        -1.78819072, -0.0742129385, -0.534911871, 0.37192291, 1.6668371, 0.0749584585, 0.674919188, 0.839969099,
        -0.0971919894, -0.511243165, -0.589661956, 0.474247187, 1.21184289, 0.111359552, -0.373548567, -0.337172121,
        0.186751217, -0.0988957509, 0.216114894, 0.664369881, 1.50278842, -0.943192244, -2.46882153, 0.226711839,
        -0.440792441, 0.60003525, -0.395036846, -0.183917075, -5.66394472, 0.271194518, 0.272476196, 0.626509845,
        -1.01296139, 0.230674997, -0.210289568, 1.55635297, 1.24095786, -0.850143015, -0.330736399, 0.197642177,
        0.179996654, -1.4259901, 0.700310826, 0.149812251, 0.463570476, -0.390995145, 0.33623901, -1.01263571,
        0.524114192, -0.42423141, -3.01410508, 1.7374897, 1.16993845, -2.07056856, 0.055507727, 0.130758271,
        0.0793126747, 1.49074471, -0.382959038, -0.890865386, -0.0381845422, -0.424962193, -0.433674276, -0.708687007,
        -0.375726342, 0.728796303, 0.181353286, 1.15931749, -0.543560326, 0.684518456, 0.618356586, -1.18480265,
        0.120278195, 0.940590143, -2.04940987, -0.216267481, 1.29770195, 1.17099833, -0.723218381, -0.278004974,
        0.519166827, -0.032496281, -1.91618407, -1.81793976, 1.379462, 0.171389461, 0.930822015, -0.276802927,
        -1.17611361, -0.779159248, 0.0792252496, -0.711527467, -2.74048352, 0.353181034, -3.76821756, -0.10598436,
        -0.702691972, 0.527826667, 0.621652067, 1.20243573, -1.51632893, -0.0339948162, 0.313036144, -0.829923809,
        -0.258761168, 0.831169724, -0.883925915, 0.951319575, -1.72474563, -1.82377386, -1.11215448, -1.22589183,
        0.177876264, 1.0588963, -0.049146276, 0.197470695, 0.0556467697, -0.598955631, 0.639007986, -0.377562076,
        0.0365857072, 0.353052527, 0.685168862, 0.495559812, -1.83929479, 0.0669759065, -0.079279393, 0.281505108,
        -0.40201211, 0.204788014, 0.127318725, -0.272549748, 1.2578032, 0.423891008, -1.67239189, -0.224743262,
        -0.122914977, 0.0461311527, -0.827723205, -1.33356726, 0.928151488, 0.340180457, -0.134445876, -0.113454908,
        0.237194568, -0.28354156, -0.476785511, 0.632535577, 0.235064209, -0.428071141, 0.542427897, 0.429924816,
        -0.0965235084, -0.40978539, 0.657142758, 0.807974219, 0.843197465, 0.679498971, 0.592249751, 0.186166942,
        -1.11913848, 0.751119554, 0.833983421, 0.426547498, 0.258929223, 1.30062342, -1.27639651, -0.110243171,
        -1.37774956, 1.5941155, 0.0743727833, 0.619498193, 0.432375789, 0.189090088, -0.0776437968, -1.38537967,
        -1.88219035, -2.12074804, 0.28353247, -0.917984009, 0.670492172, -1.24591792, 0.198109329, -0.0858308375,
        -0.665193379, -6.21732998, 0.451702207, 0.430088878, -2.93923783, -1.56569695, -0.208328083, -2.1741004,
        -0.469757497, -0.278509915, 0.41993162, -5.45535851, -0.0532281585, 0.294080824, -1.00957227, 0.367577255,
        -0.211824805, -0.651605546, 0.176654845, -1.00415099, -0.887739301, -0.440498203, 1.61846387, 0.387715191,
        0.359733611, -0.0538534857, 0.503366649, -1.03566134, -0.287378132, 0.0445584394, 0.454657972, -0.963860333,
        0.000619392726, 0.887044489, 0.605396867, 0.64468199, 1.39506638, -1.54379189, 0.424591243, 0.939089417,
        0.405189604, -1.09682178, 0.504191041, 0.381209165, 0.189687863, 0.244333714, 0.207768023, -0.389045089,
        0.0727287233, -0.364311218, 0.375694394, 0.0770283118, -0.905006289, -0.0864821151, 0.130754799, 0.404826969,
        -0.0551887266, 0.158511668, -1.19696712, -1.56820095, -0.476809591, -1.44664335, -0.594673276, 1.08765924,
        -1.16053748, 1.7783705, -1.77282119, -0.196318716, 2.42795277, -0.618356943, -1.83566821, -1.41469121,
        0.278005004, -0.922002792, -0.157025084, -0.449654609, 0.24012506, 0.996765316, -0.0430529788, 0.0169460084,
        0.904610276, -1.33751631, -1.39957619, 0.501820207, 2.26776409, -1.9069109, -0.00746337557, 0.943411469,
        0.138054818, 0.33266902, -0.166296378, -0.0927490145, 1.0626483, -0.968808532, 0.570455313, 0.139234811,
        -1.16768682, -2.41141081, -0.55752331, -0.855949104, -0.914333105, 1.70192766, -0.740169942, -0.737824202,
        -1.01496518, -0.366469532, -0.394125015, -2.35927248, -1.47800946, -2.35603666, -2.05741906, -1.21493828,
        -2.3269217, 0.690293908, -2.27494478, -0.150201276, 0.211211637, 0.201485202, 1.24441183, -1.18514025,
        0.559674203,
    )
    PARAMS = None
    PATTERN_CACHE = {}
    ADD_CACHE = {}
    BOOK_LINES = ('f5', 'f5d6', 'f5d6c3g5', 'f5d6c3g5c6c5', 'f5d6c3g5c6c5c4b6', 'f5d6c3g5c6c5c4b6f6f4', 'f5d6c3g5c6c5c4b6f6f4e6d7', 'f5d6c3g5c6c5c4b6f6f4e6d7c7g6', 'f5d6c3g5c6c5c4b6f6f4e6d7c7g6d8b5', 'f5d6c3g5c6c5c4b6f6f4e6d7c7g6d8b5e7b3', 'f5d6c3g5c6c5c4b6f6f4e6d7c7g6d8b5e7b3a6e3', 'f5d6c3g5c6c5c4b6f6f4e6d7c7g6d8b5e7b3a6e3a5d3', 'f5d6c3g5f6d3', 'f5d6c3g5f6d3e3c2', 'f5d6c3g5f6d3e3c2c1e6', 'f5d6c3g5f6d3e3c2c1e6f4f3', 'f5d6c3g5f6d3e3c2c1e6f4f3f2g4', 'f5d6c3g5f6d3e3c2c1e6f4f3f2g4g6d2', 'f5d6c3g5f6d3e3c2c1e6f4f3f2g4g6d2h3h4', 'f5d6c3g5f6d3e3c2c1e6f4f3f2g4g6d2h3h4h5f7', 'f5d6c3g5f6d3e3c2c1e6f4f3f2g4g6d2h3h4h5f7e7g3', 'f5d6c3g5g6d3', 'f5d6c3g5g6d3c4e3', 'f5d6c3g5g6d3c4e3f3b4', 'f5d6c3g5g6d3c4e3f3b4f6e6', 'f5d6c3g5g6d3c4e3f3b4f6e6f4g4', 'f5d6c3g5g6d3c4e3f3b4f6e6f4g4h4h5', 'f5d6c3g5g6d3c4e3f3b4f6e6f4g4h4h5h6g3', 'f5d6c3g5g6d3c4e3f3b4f6e6f4g4h4h5h6g3h3f7', 'f5d6c3g5g6d3c4e3f3b4f6e6f4g4h4h5h6g3h3f7f8c2', 'f5d6c4b3', 'f5d6c4b3b4f4', 'f5d6c4b3b4f4f6g5', 'f5d6c4b3b4f4f6g5f3e7', 'f5d6c4b3b4f4f6g5f3e7c5e6', 'f5d6c4b3b4f4f6g5f3e7c5e6c3g4', 'f5d6c4b3b4f4f6g5f3e7c5e6c3g4c6g3', 'f5d6c4b3b4f4f6g5f3e7c5e6c3g4c6g3h3e3', 'f5d6c4b3b4f4f6g5f3e7c5e6c3g4c6g3h3e3f2b6', 'f5d6c4b3b4f4f6g5f3e7c5e6c3g4c6g3h3e3f2b6h4d3', 'f5d6c5b4', 'f5d6c5b4d7e7', 'f5d6c5b4d7e7c7d8', 'f5d6c5b4d7e7c7d8c3d3', 'f5d6c5b4d7e7c7d8c3d3c4b3', 'f5d6c5b4d7e7c7d8c3d3c4b3d2e2', 'f5d6c5b4d7e7c7d8c3d3c4b3d2e2c2e3', 'f5d6c5b4d7e7c7d8c3d3c4b3d2e2c2e3f4f2', 'f5d6c5b4d7e7c7d8c3d3c4b3d2e2c2e3f4f2c6b5', 'f5d6c5b4d7e7c7d8c3d3c4b3d2e2c2e3f4f2c6b5f3c8', 'f5d6c4', 'f5d6c4b3b4', 'f5d6c4b3b4f4f6', 'f5d6c4b3b4f4f6g5f3', 'f5d6c4b3b4f4f6g5f3e7c5', 'f5d6c4b3b4f4f6g5f3e7c5e6c3', 'f5d6c4b3b4f4f6g5f3e7c5e6c3g4c6', 'f5d6c4b3b4f4f6g5f3e7c5e6c3g4c6g3h3', 'f5d6c4b3b4f4f6g5f3e7c5e6c3g4c6g3h3e3f2', 'f5d6c4b3b4f4f6g5f3e7c5e6c3g4c6g3h3e3f2b6h4', 'f5d6c4b3b4f4f6g5f3e7c5e6c3g4c6g3h3e3f2b6h4d3e2', 'f5d6c4d3c3', 'f5d6c4d3c3b3d2', 'f5d6c4d3c3b3d2e1b5', 'f5d6c4d3c3b3d2e1b5c5b4', 'f5d6c4d3c3b3d2e1b5c5b4e3c2', 'f5d6c4d3c3b3d2e1b5c5b4e3c2a4c6', 'f5d6c4d3c3b3d2e1b5c5b4e3c2a4c6d1e2', 'f5d6c4d3c3b3d2e1b5c5b4e3c2a4c6d1e2c7b6', 'f5d6c4d3c3b3d2e1b5c5b4e3c2a4c6d1e2c7b6f1e6', 'f5d6c4d3c3b3d2e1b5c5b4e3c2a4c6d1e2c7b6f1e6f3f2', 'f5d6c4d3c3f4f6', 'f5d6c4d3c3f4f6f3e6', 'f5d6c4d3c3f4f6f3e6e7f7', 'f5d6c4d3c3f4f6f3e6e7f7c5b6', 'f5d6c4d3c3f4f6f3e6e7f7c5b6g5e3', 'f5d6c4d3c3f4f6f3e6e7f7c5b6g5e3d7c6', 'f5d6c4d3c3f4f6f3e6e7f7c5b6g5e3d7c6e2g4', 'f5d6c4d3c3f4f6f3e6e7f7c5b6g5e3d7c6e2g4h3d2', 'f5d6c4d3c3f4f6f3e6e7f7c5b6g5e3d7c6e2g4h3d2g3f1', 'f5d6c4d3c3f4f6f3e6e7f7c5b6g6e3', 'f5d6c4d3c3f4f6f3e6e7f7c5b6g6e3e2f1', 'f5d6c4d3c3f4f6f3e6e7f7c5b6g6e3e2f1d1g5', 'f5d6c4d3c3f4f6f3e6e7f7c5b6g6e3e2f1d1g5c6d8', 'f5d6c4d3c3f4f6f3e6e7f7c5b6g6e3e2f1d1g5c6d8g4h6', 'f5d6c4d3c3f4f6b4c2', 'f5d6c4d3c3f4f6b4c2f3e3', 'f5d6c4d3c3f4f6b4c2f3e3e2c6', 'f5d6c4d3c3f4f6b4c2f3e3e2c6f2c5', 'f5d6c4d3c3f4f6b4c2f3e3e2c6f2c5e6d2', 'f5d6c4d3c3f4f6b4c2f3e3e2c6f2c5e6d2g4d7', 'f5d6c4d3c3f4f6b4c2f3e3e2c6f2c5e6d2g4d7b3g5', 'f5d6c4d3c3f4f6b4c2f3e3e2c6f2c5e6d2g4d7b3g5c8h4', 'f5d6c4d3c3f4f6g5e3', 'f5d6c4d3c3f4f6g5e3f3g6', 'f5d6c4d3c3f4f6g5e3f3g6e2h5', 'f5d6c4d3c3f4f6g5e3f3g6e2h5c5g4', 'f5d6c4d3c3f4f6g5e3f3g6e2h5c5g4g3f2', 'f5d6c4d3c3b5b4', 'f5d6c4d3c3b5b4f4c5', 'f5d6c4d3c3b5b4f4c5a4b3', 'f5d6c4d3c3b5b4f4c5a4b3d2a6', 'f5d6c4d3c3b5b4f4c5a4b3d2a6a3e3', 'f5d6c4d3c3b5b4f4c5a4b3d2a6a3e3f3g4', 'f5d6c4d3c3b5b4f4c5a4b3d2a6a3e3f3g4e6f6', 'f5d6c4d3c3b5b4f4c5a4b3d2a6a3e3f3g4e6f6g3e2', 'f5d6c4d3c3b5b4f4c5a4b3d2a6a3e3f3g4e6f6g3e2c2f2', 'f5d6c4g5f6', 'f5d6c4g5f6f4f3', 'f5d6c4g5f6f4f3d3c3', 'f5d6c4g5f6f4f3d3c3g6e3', 'f5d6c4g5f6f4f3d3c3g6e3e6h5', 'f5d6c4g5f6f4f3d3c3g6e3e6h5d2e2', 'f5d6c4g5f6f4f3d3c3g6e3e6h5d2e2c2c6', 'f5d6c4g5f6f4f3d3c3g6e3e6h5d2e2c2c6c5b6', 'f5d6c4g5f6f4f3d3c3g6e3e6h5d2e2c2c6c5b6b4b3', 'f5d6c4g5f6f4f3d3c3g6e3e6h5d2e2c2c6c5b6b4b3c7a4', 'f5f6e6', 'f5f6e6f4g6', 'f5f6e6f4g6c5f3', 'f5f6e6f4g6c5f3g4e3', 'f5f6e6f4g6c5f3g4e3d6g5', 'f5f6e6f4g6c5f3g4e3d6g5g3c3', 'f5f6e6f4g6c5f3g4e3d6g5g3c3h5c4', 'f5f6e6f4g6c5f3g4e3d6g5g3c3h5c4d7h6', 'f5f6e6f4g6c5f3g4e3d6g5g3c3h5c4d7h6h7h3', 'f5f6e6f4g6c5f3g4e3d6g5g3c3h5c4d7h6h7h3f7e7', 'f5f6e6f4g6c5f3g4e3d6g5g3c3h5c4d7h6h7h3f7e7f8h4', 'f5f6e6f4g6c5f3g5d6', 'f5f6e6f4g6c5f3g5d6e3h4', 'f5f6e6f4g6c5f3g5d6e3h4g3g4', 'f5f6e6f4g6c5f3g5d6e3h4g3g4h6e2', 'f5f6e6f4g6c5f3g5d6e3h4g3g4h6e2d3h5', 'f5f6e6f4g6c5f3g5d6e3h4g3g4h6e2d3h5h3c6', 'f5f6e6f4g6c5f3g5d6e3h4g3g4h6e2d3h5h3c6e7f2', 'f5f6e6f4g6c5f3g5d6e3h4g3g4h6e2d3h5h3c6e7f2c4d2', 'f5f6e6f4g6d6g4', 'f5f6e6f4g6d6g4g5h4', 'f5f6e6f4g6d6g4g5h4e7f3', 'f5f6e6f4g6d6g4g5h4e7f3h6f7', 'f5f6e6f4g6d6g4g5h4e7f3h6f7e8f8', 'f5f6e6f4g6d6g4g5h4e7f3h6f7e8f8g8d3', 'f5f6e6f4g6d6g4g5h4e7f3h6f7e8f8g8d3h5h7', 'f5f6e6f4g6d6g4g5h4e7f3h6f7e8f8g8d3h5h7e3c5', 'f5f6e6f4g6d6g4g5h4e7f3h6f7e8f8g8d3h5h7e3c5c4g3', 'f5f6e6d6f7', 'f5f6e6d6f7e3c6', 'f5f6e6d6f7e3c6e7f4', 'f5f6e6d6f7e3c6e7f4c5d8', 'f5f6e6d6f7e3c6e7f4c5d8c7d7', 'f5f6e6d6f7e3c6e7f4c5d8c7d7f8b5', 'f5f6e6d6f7e3c6e7f4c5d8c7d7f8b5c4e8', 'f5f6e6d6f7e3c6e7f4c5d8c7d7f8b5c4e8c8f3', 'f5f6e6d6f7e3c6e7f4c5d8c7d7f8b5c4e8c8f3g5b6', 'f5f6e6d6f7e3c6e7f4c5d8c7d7f8b5c4e8c8f3g5b6d3b4', 'f5f6e6d6f7f4d7', 'f5f6e6d6f7f4d7e7d8', 'f5f6e6d6f7f4d7e7d8g5c6', 'f5f6e6d6f7f4d7e7d8g5c6f8g6', 'f5f6e6d6f7f4d7e7d8g5c6f8g6h5h6', 'f5f6e6d6f7f4d7e7d8g5c6f8g6h5h6h7c4', 'f5f6e6d6f7f4d7e7d8g5c6f8g6h5h6h7c4e8g8', 'f5f6e6d6f7f4d7e7d8g5c6f8g6h5h6h7c4e8g8c5e3', 'f5f6e6d6f7f4d7e7d8g5c6f8g6h5h6h7c4e8g8c5e3d3c7')
    BOOK_CACHE = None

    def next_move(self, board: Board) -> Move:
        moves = self.get_moves(board)
        if not moves:
            return None

        book_move = self._book_move(board)
        if book_move in moves:
            return book_move

        best_move = moves[0]
        best_score = float("-inf")
        alpha = float("-inf")
        beta = float("inf")

        for move in moves:
            next_board = self._apply_move(board, move, self.color)
            score = -self._negascout(
                next_board,
                depth=self.SEARCH_DEPTH - 1,
                current_color=self._opponent_of(self.color),
                alpha=-beta,
                beta=-alpha,
            )
            if score > best_score:
                best_score = score
                best_move = move
            alpha = max(alpha, best_score)

        return best_move

    def _negascout(
        self,
        board: Board,
        depth: int,
        current_color: Cell,
        alpha: float,
        beta: float,
        allow_probcut: bool = True,
    ) -> float:
        if depth == 0:
            return self._evaluate_for_color(board, current_color)

        moves = self._legal_moves(board, current_color)
        next_color = self._opponent_of(current_color)

        if not moves:
            if not self._legal_moves(board, next_color):
                return self._evaluate_for_color(board, current_color)
            return -self._negascout(board, depth, next_color, -beta, -alpha, allow_probcut)

        if allow_probcut and depth >= self.PROBCUT_MIN_DEPTH:
            cut_score = self._probcut(board, depth, current_color, alpha, beta)
            if cut_score is not None:
                return cut_score

        if depth >= 2 and len(moves) > 1:
            moves.sort(
                key=lambda move: self._evaluate_for_color(
                    self._apply_move(board, move, current_color), current_color
                ),
                reverse=True,
            )

        search_window = beta
        best_score = float("-inf")

        for index, move in enumerate(moves):
            next_board = self._apply_move(board, move, current_color)
            score = -self._negascout(
                next_board, depth - 1, next_color, -search_window, -alpha, allow_probcut
            )

            if alpha < score < beta and index > 0 and depth > 1:
                score = -self._negascout(
                    next_board, depth - 1, next_color, -beta, -score, allow_probcut
                )

            best_score = max(best_score, score)
            alpha = max(alpha, score)
            if alpha >= beta:
                break
            search_window = alpha + 1

        return best_score

    def _probcut(self, board: Board, depth: int, current_color: Cell, alpha: float, beta: float):
        margin = self.PROBCUT_MARGIN + 0.02 * depth
        estimate = self._evaluate_for_color(board, current_color)
        if estimate >= beta + margin:
            return beta
        if estimate <= alpha - margin:
            return alpha
        if depth < self.PROBCUT_MIN_DEPTH:
            return None

        if depth < len(self.PROBCUT_SHALLOW_DEPTHS):
            probe_depth = self.PROBCUT_SHALLOW_DEPTHS[depth]
        else:
            probe_depth = max(1, depth - 8)
        if probe_depth <= 0:
            return None
        if estimate + margin >= beta:
            high = beta + margin
            high_score = self._negascout(
                board, probe_depth, current_color, high - 0.001, high, allow_probcut=False
            )
            if high_score >= high:
                return beta

        if estimate - margin <= alpha:
            low = alpha - margin
            low_score = self._negascout(
                board, probe_depth, current_color, low, low + 0.001, allow_probcut=False
            )
            if low_score <= low:
                return alpha

        return None

    def _evaluate_for_color(self, board: Board, color: Cell) -> float:
        score = self._evaluate_black_perspective(board)
        if color == Cell.BLACK:
            return score
        return -score

    def _evaluate_black_perspective(self, board: Board) -> float:
        flat_board = self._flatten_board(board)
        group_outputs = []
        for name, patterns in self.PATTERN_INDEXES.items():
            group_sum = 0.0
            for pattern in patterns:
                key = self._pattern_key(flat_board, pattern)
                group_sum += self._pattern_value(name, key)
            group_outputs.append(group_sum)

        add_value = self._add_value(self._additional_key(board))
        values = group_outputs + [add_value]
        final_dense, final_bias = self._params()[2]
        return final_bias + sum(values[i] * final_dense[i] for i in range(4))

    @classmethod
    def _params(cls):
        if cls.PARAMS is not None:
            return cls.PARAMS

        weights = cls.WEIGHTS
        pos = 0

        def take():
            nonlocal pos
            value = weights[pos]
            pos += 1
            return value

        patterns = {}
        for name, size in (("diagonal8", 8), ("edge2X", 10), ("triangle", 10)):
            dense0 = tuple(tuple(take() for _ in range(size * 2)) for _ in range(16))
            bias0 = tuple(take() for _ in range(16))
            dense1 = tuple(tuple(take() for _ in range(16)) for _ in range(16))
            bias1 = tuple(take() for _ in range(16))
            dense2 = tuple(take() for _ in range(16))
            bias2 = take()
            patterns[name] = (size, dense0, bias0, dense1, bias1, dense2, bias2)

        add_dense0 = tuple(tuple(take() for _ in range(3)) for _ in range(8))
        add_bias0 = tuple(take() for _ in range(8))
        add_dense1 = tuple(take() for _ in range(8))
        add_bias1 = take()
        final_dense = tuple(take() for _ in range(4))
        final_bias = take()
        cls.PARAMS = (patterns, (add_dense0, add_bias0, add_dense1, add_bias1), (final_dense, final_bias))
        return cls.PARAMS

    @classmethod
    def _pattern_value(cls, name: str, key: int) -> float:
        cache_key = (name, key)
        cached = cls.PATTERN_CACHE.get(cache_key)
        if cached is not None:
            return cached

        size, dense0, bias0, dense1, bias1, dense2, bias2 = cls._params()[0][name]
        arr = [0.0] * (size * 2)
        n = key
        for i in range(size - 1, -1, -1):
            digit = n % 3
            n //= 3
            if digit == 0:
                arr[i] = 1.0
            elif digit == 1:
                arr[size + i] = 1.0

        hidden0 = []
        for out_i in range(16):
            value = bias0[out_i]
            row = dense0[out_i]
            for in_i in range(size * 2):
                value += arr[in_i] * row[in_i]
            hidden0.append(cls._leaky_relu(value))

        result = bias2
        for out_i in range(16):
            value = bias1[out_i]
            row = dense1[out_i]
            for in_i in range(16):
                value += hidden0[in_i] * row[in_i]
            result += cls._leaky_relu(value) * dense2[out_i]

        result = cls._leaky_relu(result)
        cls.PATTERN_CACHE[cache_key] = result
        return result

    @classmethod
    def _add_value(cls, key: int) -> float:
        cached = cls.ADD_CACHE.get(key)
        if cached is not None:
            return cached

        tmp = key
        sur1 = tmp % 51
        tmp //= 51
        sur0 = tmp % 51
        mobility = tmp // 51 - 30
        arr = (mobility / 30.0, (sur0 - 15.0) / 15.0, (sur1 - 15.0) / 15.0)
        dense0, bias0, dense1, bias1 = cls._params()[1]

        hidden = []
        for out_i in range(8):
            value = bias0[out_i]
            row = dense0[out_i]
            for in_i in range(3):
                value += arr[in_i] * row[in_i]
            hidden.append(cls._leaky_relu(value))

        result = bias1
        for i in range(8):
            result += hidden[i] * dense1[i]
        result = cls._leaky_relu(result)
        cls.ADD_CACHE[key] = result
        return result

    @staticmethod
    def _leaky_relu(value: float) -> float:
        if value >= 0.0:
            return value
        return 0.01 * value

    def _pattern_key(self, flat_board: list[Cell], pattern: tuple[int, ...]) -> int:
        key = 0
        for index in pattern:
            cell = flat_board[index]
            if cell == Cell.BLACK:
                digit = 0
            elif cell == Cell.WHITE:
                digit = 1
            else:
                digit = 2
            key = key * 3 + digit
        return key

    def _additional_key(self, board: Board) -> int:
        mobility = len(self._legal_moves(board, Cell.BLACK)) - len(self._legal_moves(board, Cell.WHITE))
        mobility = max(-30, min(30, mobility))
        surround_black, surround_white = self._surround_counts(board)
        surround_black = max(0, min(50, surround_black))
        surround_white = max(0, min(50, surround_white))
        return ((mobility + 30) * 51 + surround_black) * 51 + surround_white

    def _surround_counts(self, board: Board) -> tuple[int, int]:
        counts = [0, 0]
        for row in self.BOARD_INDEXES:
            for col in self.BOARD_INDEXES:
                cell = board[row][col]
                if cell == Cell.EMPTY:
                    continue
                index = 0 if cell == Cell.BLACK else 1
                for delta_row, delta_col in self.DIRECTIONS:
                    r = row + delta_row
                    c = col + delta_col
                    if 0 <= r < 8 and 0 <= c < 8 and board[r][c] == Cell.EMPTY:
                        counts[index] += 1
        return counts[0], counts[1]

    def _flatten_board(self, board: Board) -> list[Cell]:
        return [cell for row in board for cell in row]

    def _legal_moves(self, board: Board, color: Cell) -> list[Move]:
        moves = []
        for row in self.BOARD_INDEXES:
            for col in self.BOARD_INDEXES:
                if board[row][col] == Cell.EMPTY and self._flips_for_color(board, row, col, color):
                    moves.append((row, col))
        return moves

    def _apply_move(self, board: Board, move: Move, color: Cell) -> Board:
        row, col = move
        next_board = [board_row[:] for board_row in board]
        next_board[row][col] = color

        for flip_row, flip_col in self._flips_for_color(board, row, col, color):
            next_board[flip_row][flip_col] = color

        return next_board

    def _flips_for_color(self, board: Board, row: int, col: int, color: Cell) -> list[Move]:
        if board[row][col] != Cell.EMPTY:
            return []

        opponent_color = self._opponent_of(color)
        flips = []

        for delta_row, delta_col in self.DIRECTIONS:
            direction_flips = []
            current_row = row + delta_row
            current_col = col + delta_col

            while (
                0 <= current_row < 8
                and 0 <= current_col < 8
                and board[current_row][current_col] == opponent_color
            ):
                direction_flips.append((current_row, current_col))
                current_row += delta_row
                current_col += delta_col

            if (
                direction_flips
                and 0 <= current_row < 8
                and 0 <= current_col < 8
                and board[current_row][current_col] == color
            ):
                flips.extend(direction_flips)

        return flips

    @classmethod
    def _book_cache(cls):
        if cls.BOOK_CACHE is not None:
            return cls.BOOK_CACHE

        cache = {}
        for line in cls.BOOK_LINES:
            prefix = line[:-2]
            move_text = line[-2:]
            for transform in range(4):
                board = cls._initial_board()
                current_color = Cell.BLACK
                for i in range(0, len(prefix), 2):
                    move = cls._coord_to_move(prefix[i:i + 2], transform)
                    board = cls._apply_move_for_color(board, move, current_color)
                    current_color = Cell.WHITE if current_color == Cell.BLACK else Cell.BLACK
                cache[(cls._board_key(board), current_color)] = cls._coord_to_move(move_text, transform)

        cls.BOOK_CACHE = cache
        return cache

    def _book_move(self, board: Board) -> Move:
        return self._book_cache().get((self._board_key(board), self.color))

    @staticmethod
    def _initial_board() -> Board:
        board = [[Cell.EMPTY for _ in range(8)] for _ in range(8)]
        board[3][3] = Cell.WHITE
        board[3][4] = Cell.BLACK
        board[4][3] = Cell.BLACK
        board[4][4] = Cell.WHITE
        return board

    @staticmethod
    def _coord_to_move(coord: str, transform: int) -> Move:
        col = ord(coord[0]) - ord("a")
        row = int(coord[1]) - 1
        if transform == 0:
            return (row, col)
        if transform == 1:
            return (col, row)
        if transform == 2:
            return (7 - row, 7 - col)
        return (7 - col, 7 - row)

    @classmethod
    def _apply_move_for_color(cls, board: Board, move: Move, color: Cell) -> Board:
        row, col = move
        next_board = [board_row[:] for board_row in board]
        next_board[row][col] = color
        opponent_color = Cell.WHITE if color == Cell.BLACK else Cell.BLACK

        for delta_row, delta_col in cls.DIRECTIONS:
            direction_flips = []
            current_row = row + delta_row
            current_col = col + delta_col
            while (
                0 <= current_row < 8
                and 0 <= current_col < 8
                and board[current_row][current_col] == opponent_color
            ):
                direction_flips.append((current_row, current_col))
                current_row += delta_row
                current_col += delta_col

            if (
                direction_flips
                and 0 <= current_row < 8
                and 0 <= current_col < 8
                and board[current_row][current_col] == color
            ):
                for flip_row, flip_col in direction_flips:
                    next_board[flip_row][flip_col] = color

        return next_board

    @staticmethod
    def _board_key(board: Board) -> tuple[int, ...]:
        values = []
        for row in board:
            for cell in row:
                if cell == Cell.BLACK:
                    values.append(1)
                elif cell == Cell.WHITE:
                    values.append(2)
                else:
                    values.append(0)
        return tuple(values)

    def _opponent_of(self, color: Cell) -> Cell:
        return Cell.WHITE if color == Cell.BLACK else Cell.BLACK

強化学習を考えたコード↓


**<font color="red">↑ 以上のコードセルを実装・提出</font>**

---

### **相手AIとの対戦**

実際に対戦してみましょう．\
ライブラリ `othellopy` では，3レベルのAIを用意しています．

* `BeginnerPlayer`：初級．ランダムに置くだけの戦略
* `IntermediatePlayer`：中級．置くと有利になる場所に優先して置く戦略
* `AdvancedPlayer`：上級．数手先の盤面まで予測する戦略

In [ ]:
from othellopy.game import OthelloGame
from othellopy.players import AdvancedPlayer
from pathlib import Path
import json
import time


EVALUATION_OUTPUT_PATH = Path("othello_ai_pattern_evaluation_results.json")


def _count_stones_from_board(board):
    black = 0
    white = 0
    for row in board:
        for cell in row:
            value = getattr(cell, "value", cell)
            if value == 1:
                black += 1
            elif value == 2:
                white += 1
    return black, white


def _get_attr(obj, names, default=None):
    for name in names:
        if isinstance(obj, dict) and name in obj:
            return obj[name]
        if hasattr(obj, name):
            value = getattr(obj, name)
            return value() if callable(value) else value
    return default


def _to_jsonable(value):
    value = getattr(value, "value", value)
    if value is None or isinstance(value, (bool, int, float, str)):
        return value
    if isinstance(value, (tuple, list)):
        return [_to_jsonable(v) for v in value]
    if isinstance(value, dict):
        return {str(k): _to_jsonable(v) for k, v in value.items()}

    attrs = {}
    for name in ("x", "y", "row", "col", "r", "c", "i", "j"):
        if hasattr(value, name):
            attrs[name] = _to_jsonable(getattr(value, name))
    if attrs:
        attrs["repr"] = repr(value)
        return attrs

    return repr(value)


def _player_name(player):
    return getattr(player, "__name__", player.__class__.__name__)


def _timed_player_class(player_cls, color_name, game_index, move_logs):
    class TimedPlayer:
        def __init__(self, *args, **kwargs):
            self._inner = player_cls(*args, **kwargs) if isinstance(player_cls, type) else player_cls
            self._color_name = color_name

        def __getattr__(self, name):
            return getattr(self._inner, name)

        def next_move(self, board):
            start = time.perf_counter()
            move = self._inner.next_move(board)
            elapsed = time.perf_counter() - start
            move_logs.append({
                "game_index": game_index,
                "turn_index": len(move_logs) + 1,
                "color": self._color_name,
                "player": _player_name(self._inner),
                "elapsed_sec": elapsed,
                "move": _to_jsonable(move),
            })
            return move

    TimedPlayer.__name__ = f"Timed{_player_name(player_cls)}"
    return TimedPlayer


def _slowest_move_from_logs(move_logs):
    if not move_logs:
        return None
    slowest = max(move_logs, key=lambda r: r["elapsed_sec"])
    return {
        **slowest,
        "elapsed_ms": slowest["elapsed_sec"] * 1000,
    }


def _result_summary(result, my_color, game=None):
    board = _get_attr(result, ("board", "final_board"), None)
    black_score = _get_attr(result, ("black_score", "black_count", "black"), None)
    white_score = _get_attr(result, ("white_score", "white_count", "white"), None)

    if isinstance(result, (tuple, list)) and len(result) >= 2:
        if isinstance(result[0], int) and isinstance(result[1], int):
            black_score, white_score = result[0], result[1]

    if game is not None:
        if board is None:
            board = _get_attr(game, ("board", "final_board"), None)
        if black_score is None:
            black_score = _get_attr(game, ("black_score", "black_count", "black"), None)
        if white_score is None:
            white_score = _get_attr(game, ("white_score", "white_count", "white"), None)

    if (black_score is None or white_score is None) and board is not None:
        black_score, white_score = _count_stones_from_board(board)

    if black_score is None or white_score is None:
        raise ValueError("Could not read game result.")

    winner = _get_attr(result, ("winner", "winner_color"), None)
    black_score = int(black_score)
    white_score = int(white_score)
    diff = black_score - white_score

    if winner is None:
        if diff > 0:
            winner = 1
        elif diff < 0:
            winner = 2
        else:
            winner = 0
    else:
        winner = getattr(winner, "value", winner)

    my_value = getattr(my_color, "value", my_color)
    if winner == 0 or black_score == white_score:
        outcome = "draw"
    elif winner == my_value:
        outcome = "win"
    else:
        outcome = "loss"

    my_score = black_score if my_value == 1 else white_score
    opp_score = white_score if my_value == 1 else black_score
    return {
        "outcome": outcome,
        "black_score": black_score,
        "white_score": white_score,
        "diff_black": diff,
        "my_score": my_score,
        "opp_score": opp_score,
        "my_diff": my_score - opp_score,
    }


def _play_many(label, black_player, white_player, my_color, games=100):
    records = []
    start = time.time()
    for i in range(games):
        move_logs = []
        game = OthelloGame(
            black_player=_timed_player_class(black_player, "black", i + 1, move_logs),
            white_player=_timed_player_class(white_player, "white", i + 1, move_logs),
        )
        result = game.play()
        record = _result_summary(result, my_color, game)
        record["game_index"] = i + 1
        record["slowest_move"] = _slowest_move_from_logs(move_logs)
        records.append(record)

    elapsed = time.time() - start
    wins = sum(r["outcome"] == "win" for r in records)
    losses = sum(r["outcome"] == "loss" for r in records)
    draws = sum(r["outcome"] == "draw" for r in records)
    diffs = [r["my_diff"] for r in records]
    my_scores = [r["my_score"] for r in records]
    opp_scores = [r["opp_score"] for r in records]
    slowest_moves = [r["slowest_move"] for r in records if r["slowest_move"] is not None]

    summary = {
        "label": label,
        "games": games,
        "wins": wins,
        "losses": losses,
        "draws": draws,
        "win_rate": wins / games,
        "avg_my_score": sum(my_scores) / games,
        "avg_opp_score": sum(opp_scores) / games,
        "avg_my_diff": sum(diffs) / games,
        "best_my_diff": max(diffs),
        "worst_my_diff": min(diffs),
        "elapsed_sec": elapsed,
        "sec_per_game": elapsed / games,
        "slowest_move_overall": max(slowest_moves, key=lambda r: r["elapsed_sec"]) if slowest_moves else None,
    }
    return summary, records


black_summary, black_records = _play_many(
    "MyPlayer as BLACK vs AdvancedPlayer",
    black_player=MyPlayer,
    white_player=AdvancedPlayer,
    my_color=1,
    games=100,
)

white_summary, white_records = _play_many(
    "MyPlayer as WHITE vs AdvancedPlayer",
    black_player=AdvancedPlayer,
    white_player=MyPlayer,
    my_color=2,
    games=100,
)

total_games = black_summary["games"] + white_summary["games"]
total_wins = black_summary["wins"] + white_summary["wins"]
total_losses = black_summary["losses"] + white_summary["losses"]
total_draws = black_summary["draws"] + white_summary["draws"]
all_records = black_records + white_records
all_diffs = [r["my_diff"] for r in all_records]
all_slowest_moves = [r["slowest_move"] for r in all_records if r["slowest_move"] is not None]

evaluation_result = {
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "summaries": [black_summary, white_summary],
    "total": {
        "games": total_games,
        "wins": total_wins,
        "losses": total_losses,
        "draws": total_draws,
        "win_rate": total_wins / total_games,
        "avg_my_diff": sum(all_diffs) / total_games,
        "best_my_diff": max(all_diffs),
        "worst_my_diff": min(all_diffs),
        "slowest_move_overall": max(all_slowest_moves, key=lambda r: r["elapsed_sec"]) if all_slowest_moves else None,
    },
    "records": {
        "black": black_records,
        "white": white_records,
    },
}

_ = EVALUATION_OUTPUT_PATH.write_text(
    json.dumps(evaluation_result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

### **結果の表示**

In [ ]:
# from othellopy.board import display_board
# # display_board: 盤面をグラフィカルに表示する関数

# print("Winner:", result.winner_name)
# print("Black:", result.black_score)
# print("White", result.white_score)
# display_board(result.board)

Winner: BLACK
Black: 37
White 27


,0,1,2,3,4,5,6,7
0,⚫️,⚫️,⚪️,⚪️,⚪️,⚪️,⚪️,⚪️
1,⚫️,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚫️
2,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️
3,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️
4,⚫️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️
5,⚫️,⚪️,⚫️,⚪️,⚫️,⚪️,⚫️,⚫️
6,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️,⚪️,⚫️
7,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️


1手ずつ詳細を確認することもできます．

In [ ]:
# for i, turn in enumerate(result.turns):
#     print(f"Turn {i}: {turn.color.name}")

#     if turn.move is None:
#         print("Move: pass")
#     else:
#         row, col = turn.move
#         print(f"Move: {row}{col}")

#     print(f"Valid moves: {turn.valid_moves}")
#     print(f"Score: BLACK {turn.black_score} - WHITE {turn.white_score}")
#     display_board(turn.board)
#     print()

# print("Winner:", result.winner_name)

Turn 0: BLACK
Move: 23
Valid moves: [(2, 3), (3, 2), (4, 5), (5, 4)]
Score: BLACK 4 - WHITE 1


,0,1,2,3,4,5,6,7
0,.,.,.,.,.,.,.,.
1,.,.,.,.,.,.,.,.
2,.,.,.,⚫️,.,.,.,.
3,.,.,.,⚫️,⚫️,.,.,.
4,.,.,.,⚫️,⚪️,.,.,.
5,.,.,.,.,.,.,.,.
6,.,.,.,.,.,.,.,.
7,.,.,.,.,.,.,.,.



Turn 1: WHITE
Move: 24
Valid moves: [(2, 2), (2, 4), (4, 2)]
Score: BLACK 3 - WHITE 3


,0,1,2,3,4,5,6,7
0,.,.,.,.,.,.,.,.
1,.,.,.,.,.,.,.,.
2,.,.,.,⚫️,⚪️,.,.,.
3,.,.,.,⚫️,⚪️,.,.,.
4,.,.,.,⚫️,⚪️,.,.,.
5,.,.,.,.,.,.,.,.
6,.,.,.,.,.,.,.,.
7,.,.,.,.,.,.,.,.



Turn 2: BLACK
Move: 35
Valid moves: [(1, 5), (2, 5), (3, 5), (4, 5), (5, 5)]
Score: BLACK 5 - WHITE 2


...


Turn 60: BLACK
Move: pass
Valid moves: []
Score: BLACK 37 - WHITE 27


,0,1,2,3,4,5,6,7
0,⚫️,⚫️,⚪️,⚪️,⚪️,⚪️,⚪️,⚪️
1,⚫️,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚫️
2,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️
3,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️
4,⚫️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️
5,⚫️,⚪️,⚫️,⚪️,⚫️,⚪️,⚫️,⚫️
6,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️,⚪️,⚫️
7,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️



Turn 61: WHITE
Move: pass
Valid moves: []
Score: BLACK 37 - WHITE 27


,0,1,2,3,4,5,6,7
0,⚫️,⚫️,⚪️,⚪️,⚪️,⚪️,⚪️,⚪️
1,⚫️,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚫️
2,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️
3,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️
4,⚫️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️
5,⚫️,⚪️,⚫️,⚪️,⚫️,⚪️,⚫️,⚫️
6,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️,⚪️,⚫️
7,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️



Winner: BLACK


### **`MyPlayer` の提出チェック**

自作した `MyPlayer` に不具合がないかを確認するためのテストを用意しています．

提出前のチェックにご利用ください．


In [ ]:
from othellopy.validation import test_player, test_player_detail
# test_player: 作成したプレイヤークラスの簡易動作チェック
# test_player_detail: 作成したプレイヤークラスの簡易動作チェック（詳細情報付き）

if test_player(MyPlayer):
    print("テストを PASS しました．")
else:
    result = test_player_detail(MyPlayer)
    for issue in result.errors:
        print(issue.code, issue.message)

テストを PASS しました．
